In [1]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import pandas as pd
import re
import yaml
import json
import os
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Load Config
def load_config(config_path="config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

config = load_config()

# Load Slang Dictionary
with open(config['preprocessing']['slang_dictionary'], 'r') as f:
    slang_dict = json.load(f)

In [3]:
# List untuk menampung data gabungan
all_dfs = []
raw_dir = config['data']['raw_dir']

for bank in config['data']['banks']:
    file_path = os.path.join(raw_dir, f"{bank}_reviews.csv")
    if os.path.exists(file_path):
        temp_df = pd.read_csv(file_path)
        # Filter rating sesuai config
        temp_df = temp_df[temp_df['score'].isin(config['data']['ratings_filter'])]
        # Tambah label bank
        temp_df['bank'] = bank
        all_dfs.append(temp_df)
        print(f"Loaded {bank}: {len(temp_df)} ulasan.")
    else:
        print(f"Warning: {file_path} tidak ditemukan.")

# Gabungkan semua bank
df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal gabungan data keluhan: {len(df)} ulasan.")
df.head()

Loaded BCAMobile: 37827 ulasan.
Loaded BRImo: 71882 ulasan.
Loaded livin_mandiri: 63048 ulasan.
Loaded Wondr_BNI: 21252 ulasan.

Total gabungan data keluhan: 194009 ulasan.


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,pajar,https://play-lh.googleusercontent.com/a/ACg8oc...,ribet banget ni apk sumpah dikir' verif dikit'...,1,0,NaN,2025-12-31 23:57:59,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2026-01-01 00:10:13,NaN,BCAMobile
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,Jessica Slycia,https://play-lh.googleusercontent.com/a/ACg8oc...,kenapa QRis ga bisa di pakai ya?? daritadi loa...,2,0,4.7.7,2025-12-31 23:22:32,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 23:33:06,4.7.7,BCAMobile
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,Bidibetico 09,https://play-lh.googleusercontent.com/a-/ALV-U...,"aplikasi nya sampah, kenapa tiba-tiba keluar t...",1,0,4.7.7,2025-12-31 23:06:58,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 23:30:30,4.7.7,BCAMobile
3,32d28ac6-c538-4749-969f-8af060915d96,ABDUL Kodir,https://play-lh.googleusercontent.com/a/ACg8oc...,bagus,3,0,4.7.7,2025-12-31 21:47:45,Terima kasih atas ulasannya. Semoga aplikasi B...,2025-12-31 22:06:01,4.7.7,BCAMobile
4,f99259b7-0d45-4422-b667-6c4fd521638b,Prr G,https://play-lh.googleusercontent.com/a/ACg8oc...,tidak ada solusi ketika ada kendala di persuli...,1,0,2.9.6,2025-12-31 21:18:21,"Mohon maaf atas ketidaknyamanan Bapak/Ibu, moh...",2025-12-31 22:04:38,2.9.6,BCAMobile


In [4]:
df.isnull().sum()

reviewId                    0
userName                    0
userImage                   0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion    41660
at                          0
replyContent             1753
repliedAt                1753
appVersion              41660
bank                        0
dtype: int64

In [5]:
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def clean_text_basic(text):
    # 1. Lowercase
    if config['preprocessing']['do_lowercase']:
        text = str(text).lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Remove Digits
    if config['preprocessing']['remove_digits']:
        text = re.sub(r'\d+', ' ', text)
    
    # 4. Remove Special Characters
    if config['preprocessing']['remove_special_chars']:
        text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 5. Slang Mapping (Normalization)
    words = text.split()
    normalized_words = [slang_dict.get(w, w) for w in words]
    text = " ".join(normalized_words)
    
    # 6. Stopword Removal
    text = stopword_remover.remove(text)
    
    # 7. Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
print("Memulai proses pembersihan teks (ini mungkin memakan waktu)... ")
df['content_cleaned'] = df['content'].apply(clean_text_basic)
df['cleaned_word_count'] = df['content_cleaned'].apply(lambda x: len(str(x).split()))

# Buang ulasan yang terlalu pendek sesuai config
min_words = config['preprocessing']['min_word_count']
df_final = df[df['cleaned_word_count'] >= min_words].copy()

print(f"Total data final setelah dibersihkan: {len(df_final)} baris")
df_final.head(10)

Memulai proses pembersihan teks (ini mungkin memakan waktu)... 
Total data final setelah dibersihkan: 179828 baris


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank,content_cleaned,cleaned_word_count
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,pajar,https://play-lh.googleusercontent.com/a/ACg8oc...,ribet banget ni apk sumpah dikir' verif dikit'...,1,0,NaN,2025-12-31 23:57:59,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2026-01-01 00:10:13,NaN,BCAMobile,ribet banget nih apk sumpah dikir verif dikit ...,22
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,Jessica Slycia,https://play-lh.googleusercontent.com/a/ACg8oc...,kenapa QRis ga bisa di pakai ya?? daritadi loa...,2,0,4.7.7,2025-12-31 23:22:32,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 23:33:06,4.7.7,BCAMobile,qris enggak di pakai daritadi loading mulu tra...,10
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,Bidibetico 09,https://play-lh.googleusercontent.com/a-/ALV-U...,"aplikasi nya sampah, kenapa tiba-tiba keluar t...",1,0,4.7.7,2025-12-31 23:06:58,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 23:30:30,4.7.7,BCAMobile,aplikasi nya sampah tiba tiba keluar terus reg...,16
4,f99259b7-0d45-4422-b667-6c4fd521638b,Prr G,https://play-lh.googleusercontent.com/a/ACg8oc...,tidak ada solusi ketika ada kendala di persuli...,1,0,2.9.6,2025-12-31 21:18:21,"Mohon maaf atas ketidaknyamanan Bapak/Ibu, moh...",2025-12-31 22:04:38,2.9.6,BCAMobile,ada solusi ada kendala persulit lempar kesana ...,15
5,1b50a316-798a-42a3-8bfe-0154853d0f4a,rinni rositass,https://play-lh.googleusercontent.com/a-/ALV-U...,masa gue setiap isi Flazz selalu gagal tapi ke...,1,0,4.7.7,2025-12-31 20:17:13,"Mohon maaf atas ketidaknyamanan Bapak/Ibu, unt...",2025-12-31 21:01:03,4.7.7,BCAMobile,masa gue isi flazz selalu gagal kedebit enggak...,13
6,6ac413fd-0745-430c-b545-da39c54ff60e,Kia Jr24,https://play-lh.googleusercontent.com/a-/ALV-U...,Mau Aktifasi di Handphone baru semakin ribet d...,1,0,4.7.7,2025-12-31 19:40:08,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 19:51:01,4.7.7,BCAMobile,mau aktifasi handphone baru semakin ribet suli...,9
7,8fce3e87-d488-4ed5-9f91-17cdae1a6b5f,Ongky Wijaya,https://play-lh.googleusercontent.com/a-/ALV-U...,"sinyal penuh, internet lancar tapi masih aj ko...",1,0,4.2.4,2025-12-31 18:54:08,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 19:11:56,4.2.4,BCAMobile,sinyal penuh internet lancar masih kode merah ...,9
8,b8568501-8111-48cc-9fd9-eefda3fdd1d6,Fariz Ardhian,https://play-lh.googleusercontent.com/a/ACg8oc...,sinyal bagus tetap biru terus..membuat lama pe...,2,0,4.7.7,2025-12-31 17:47:04,Mohon maaf atas ketidaknyamanan Bapak/Ibu. Moh...,2025-12-31 18:09:45,4.7.7,BCAMobile,sinyal bagus tetap biru terus membuat lama pem...,8
9,06df10c9-69d4-4589-bd3d-505443d4fd53,Fath Kha,https://play-lh.googleusercontent.com/a-/ALV-U...,saya mau login menggunakan pulsa kode selalu t...,1,0,4.7.7,2025-12-31 16:41:22,"Mohon maaf atas ketidaknyamanan Bapak/Ibu, moh...",2025-12-31 17:18:13,4.7.7,BCAMobile,mau login menggunakan pulsa kode selalu keluar,7
10,7cdf7f58-769d-4779-87f6-a0184e893fac,Andi Cahya,https://play-lh.googleusercontent.com/a-/ALV-U...,kenapa aplikasi ini sering sekarat,1,0,4.7.7,2025-12-31 14:17:32,"Mohon maaf atas ketidaknyamanan Bapak/Ibu, unt...",2025-12-31 14:26:51,4.7.7,BCAMobile,aplikasi sering sekarat,3


In [7]:
# Export cleaned master dataframe to CSV
output_path = config['data']['processed_file']
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_export = df_final[['content_cleaned', 'score', 'at', 'bank']].rename(columns={'content_cleaned': 'content'})
df_export.to_csv(output_path, index=False, encoding='utf-8')

print(f'Exported cleaned master dataframe to: {output_path}')

Exported cleaned master dataframe to: data/processed/cleaned_reviews.csv
